# Silver to Gold - Biblioteca Dataset

Lee las tablas Delta de la capa silver (ADLS Gen2), exporta las 4 tablas relacionales
a Azure SQL Database y genera la tabla desnormalizada `dashboard_actividad_usuarios`.

In [ ]:
STORAGE_ACCOUNT_NAME = "stadatademo2026"
SILVER_CONTAINER = "silver"
SILVER_BASE_PATH = f"abfss://{SILVER_CONTAINER}@{STORAGE_ACCOUNT_NAME}.dfs.core.windows.net"

JDBC_DRIVER = "com.microsoft.sqlserver.jdbc.SQLServerDriver"
TABLE_NAMES = ["autores", "libros", "usuarios", "prestamos"]

In [ ]:
storage_account_key = dbutils.secrets.get(scope="data-demo-scope", key="storage-account-key")
jdbc_connection_string = dbutils.secrets.get(scope="data-demo-scope", key="sql-connection-string")

spark.conf.set(f"fs.azure.account.key.{STORAGE_ACCOUNT_NAME}.dfs.core.windows.net", storage_account_key)

## 1. Lectura de tablas Delta desde la capa Silver

In [ ]:
silver_dataframes = {}

for table_name in TABLE_NAMES:
    silver_path = f"{SILVER_BASE_PATH}/{table_name}"
    silver_dataframes[table_name] = spark.read.format("delta").load(silver_path)

silver_authors = silver_dataframes["autores"]
silver_books = silver_dataframes["libros"]
silver_users = silver_dataframes["usuarios"]
silver_loans = silver_dataframes["prestamos"]

for table_name, dataframe in silver_dataframes.items():
    print(f"{table_name}: {dataframe.count()} registros")

## 2. Exportar tablas relacionales a Azure SQL Database

In [ ]:
def write_to_sql(dataframe, table_name, connection_string):
    dataframe.write.format("jdbc") \
        .option("url", connection_string) \
        .option("dbtable", table_name) \
        .option("driver", JDBC_DRIVER) \
        .mode("overwrite") \
        .save()


for table_name, dataframe in silver_dataframes.items():
    write_to_sql(dataframe, table_name, jdbc_connection_string)
    print(f"Exportado a SQL: {table_name} ({dataframe.count()} registros)")

## 3. Tabla desnormalizada: dashboard_actividad_usuarios

In [ ]:
from pyspark.sql.functions import col, count, when, max, avg, datediff, first, mode

is_returned = col("estado") == "devuelto"
is_overdue = col("estado") == "vencido"
is_active = col("estado") == "prestado"

loans_with_duration = silver_loans.withColumn(
    "loan_duration_days", datediff(col("fecha_devolucion"), col("fecha_prestamo"))
)

loans_with_genre = loans_with_duration.join(silver_books, loans_with_duration.libro_id == silver_books.id, "left") \
    .select(loans_with_duration["*"], silver_books.genero)

In [ ]:
user_activity = loans_with_genre.groupBy("usuario_id").agg(
    count("*").alias("total_prestamos"),
    count(when(is_active, True)).alias("prestamos_activos"),
    count(when(is_overdue, True)).alias("prestamos_vencidos"),
    count(when(is_returned, True)).alias("prestamos_devueltos"),
    avg(when(is_returned, col("loan_duration_days"))).alias("promedio_dias_tenencia"),
    max(col("fecha_prestamo")).alias("fecha_ultimo_prestamo"),
    mode(col("genero")).alias("genero_mas_leido")
)

In [ ]:
dashboard_user_activity = user_activity.join(silver_users, user_activity.usuario_id == silver_users.id, "inner") \
    .select(
        silver_users.id.alias("usuario_id"),
        silver_users.nombre.alias("nombre_usuario"),
        silver_users.email,
        silver_users.tipo_socio,
        silver_users.fecha_registro,
        user_activity.total_prestamos,
        user_activity.prestamos_activos,
        user_activity.prestamos_devueltos,
        user_activity.prestamos_vencidos,
        user_activity.promedio_dias_tenencia,
        user_activity.fecha_ultimo_prestamo,
        user_activity.genero_mas_leido
    )

print(f"dashboard_actividad_usuarios: {dashboard_user_activity.count()} registros")
dashboard_user_activity.show(10, truncate=False)

In [ ]:
write_to_sql(dashboard_user_activity, "dashboard_actividad_usuarios", jdbc_connection_string)
print("Exportado a SQL: dashboard_actividad_usuarios")

## 4. Verificacion

In [ ]:
EXPORTED_TABLES = TABLE_NAMES + ["dashboard_actividad_usuarios"]

print("Verificacion de tablas en SQL:")
print(f"{'Tabla':<35} {'Registros':>10}")
print("-" * 47)

for exported_table in EXPORTED_TABLES:
    sql_dataframe = spark.read.format("jdbc") \
        .option("url", jdbc_connection_string) \
        .option("dbtable", exported_table) \
        .option("driver", JDBC_DRIVER) \
        .load()
    print(f"{exported_table:<35} {sql_dataframe.count():>10}")